In [ ]:

!pip install -q kaggle

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"kofiboadisarfo","key":"1fdca5f8b8c4444d8c10244e4b29ac88"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:

!kaggle datasets download -d abdallahalidev/plantvillage-dataset
!unzip -q plantvillage-dataset.zip -d plantvillage_data


Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.04G/2.04G [00:24<00:00, 89.3MB/s]



In [ ]:
import os
import shutil

source_dir = 'plantvillage_data/plantvillage dataset/color'

Crop_data = 'Crop_data'

os.makedirs(Crop_data, exist_ok=True)

Study_crops = ['Tomato', 'Corn_(maize)', 'Potato', 'Pepper']

for folder_name in os.listdir(source_dir):
    if any(crop in folder_name for crop in Study_crops):

        src_path = os.path.join(source_dir, folder_name)
        dst_path = os.path.join(Crop_data, folder_name)

        if not os.path.exists(dst_path):
            shutil.copytree(src_path, dst_path)
            print(f"Copied: {folder_name}")

print("Data filtering complete!")

Copied: Tomato___Late_blight
Copied: Tomato___Target_Spot
Copied: Potato___Early_blight
Copied: Corn_(maize)___Common_rust_
Copied: Tomato___Leaf_Mold
Copied: Tomato___healthy
Copied: Pepper,_bell___Bacterial_spot
Copied: Tomato___Tomato_mosaic_virus
Copied: Pepper,_bell___healthy
Copied: Corn_(maize)___Northern_Leaf_Blight
Copied: Tomato___Spider_mites Two-spotted_spider_mite
Copied: Tomato___Septoria_leaf_spot
Copied: Potato___healthy
Copied: Corn_(maize)___healthy
Copied: Potato___Late_blight
Copied: Tomato___Tomato_Yellow_Leaf_Curl_Virus
Copied: Tomato___Bacterial_spot
Copied: Tomato___Early_blight
Copied: Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
Data filtering complete!


In [ ]:
os.listdir("Crop_data")

['Tomato___Late_blight',
 'Tomato___Target_Spot',
 'Potato___Early_blight',
 'Corn_(maize)___Common_rust_',
 'Tomato___Leaf_Mold',
 'Tomato___healthy',
 'Pepper,_bell___Bacterial_spot',
 'Tomato___Tomato_mosaic_virus',
 'Pepper,_bell___healthy',
 'Corn_(maize)___Northern_Leaf_Blight',
 'Tomato___Spider_mites Two-spotted_spider_mite',
 'Tomato___Septoria_leaf_spot',
 'Potato___healthy',
 'Corn_(maize)___healthy',
 'Potato___Late_blight',
 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
 'Tomato___Bacterial_spot',
 'Tomato___Early_blight',
 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot']

In [ ]:
# split-folders helper
!pip install split-folders

import splitfolders

# Split your filtered dataset into train (80%), val (10%), test (10%)
splitfolders.ratio(
    "Crop_data",       # Your filtered folder name
    output="data_split",      # New folder with split subdirectories
    seed=42,
    ratio=(0.8, 0.1, 0.1)
)

print("✅ Data successfully split into train, val, and test folders!")


Copying files: 26639 files [00:14, 1849.38 files/s]

✅ Data successfully split into train, val, and test folders!


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Rescale image pixels (from 0-255 down to 0.0-1.0)
datagen = ImageDataGenerator(rescale=1./255)

IMG_SIZE = (128, 128) # Resizing images to 128x128 keeps training fast for Thursday's demo
BATCH_SIZE = 32

# Create the training stream
train_generator = datagen.flow_from_directory(
    'data_split/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# Create the validation stream
val_generator = datagen.flow_from_directory(
    'data_split/val',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# Create the testing stream
test_generator = datagen.flow_from_directory(
    'data_split/test',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)


Found 21303 images belonging to 19 classes.
Found 2657 images belonging to 19 classes.
Found 2679 images belonging to 19 classes.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Get the number of crop classes automatically
num_classes = train_generator.num_classes

# Build a simple baseline CNN architecture
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation='softmax') # Outputs probability for each crop disease class
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model for 5 epochs to establish a quick baseline
history = model.fit(
    train_generator,
    epochs=5,
    validation_data=val_generator
)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
666/666 ━━━━━━━━━━━━━━━━━━━━ 616s 921ms/step - accuracy: 0.7093 - loss: 0.9325 - val_accuracy: 0.8427 - val_loss: 0.4691
Epoch 2/5
666/666 ━━━━━━━━━━━━━━━━━━━━ 630s 945ms/step - accuracy: 0.8813 - loss: 0.3550 - val_accuracy: 0.8822 - val_loss: 0.3467
Epoch 3/5
666/666 ━━━━━━━━━━━━━━━━━━━━ 602s 904ms/step - accuracy: 0.9323 - loss: 0.2010 - val_accuracy: 0.8833 - val_loss: 0.3766
Epoch 4/5
666/666 ━━━━━━━━━━━━━━━━━━━━ 603s 905ms/step - accuracy: 0.9645 - loss: 0.1048 - val_accuracy: 0.8630 - val_loss: 0.4934
Epoch 5/5
666/666 ━━━━━━━━━━━━━━━━━━━━ 595s 893ms/step - accuracy: 0.9756 - loss: 0.0734 - val_accuracy: 0.8803 - val_loss: 0.4853


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# 1. Get the overall accuracy on the test set
print("Evaluating baseline model on unseen test data...")
test_loss, test_acc = model.evaluate(test_generator)
print(f"\nOverall Test Accuracy: {test_acc * 100:.2f}%\n")

# 2. Get predictions for the confusion matrix and metrics
# IMPORTANT: Reset the generator to ensure it starts from the very first image
test_generator.reset()
print("Generating predictions...")
predictions = model.predict(test_generator)

# Convert the raw probabilities into final class predictions
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

# 3. Generate Precision, Recall, and F1-Score
class_labels = list(test_generator.class_indices.keys())
print("\n--- Classification Report ---")
print(classification_report(y_true, y_pred, target_names=class_labels))

# 4. Plot the Confusion Matrix
print("\n--- Confusion Matrix ---")
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(14, 12)) # Make the chart large enough to read the crop names
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels)
plt.title('Baseline CNN - Confusion Matrix', fontsize=16)
plt.ylabel('Actual Crop/Disease', fontsize=12)
plt.xlabel('Predicted Crop/Disease', fontsize=12)
plt.xticks(rotation=90) # Rotate the labels so they don't overlap
plt.tight_layout()
plt.show()